# Wavelet-Based Image Compression

In [1]:
!pip install PyWavelets scikit-image -q

In [2]:
import os
import tarfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from google.colab import drive

seed_orbit = 413
random.seed(seed_orbit)
np.random.seed(seed_orbit)
torch.manual_seed(seed_orbit)
torch.cuda.manual_seed_all(seed_orbit)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

run_device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", run_device)

device: cpu


## Dataset

In [4]:
drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/EE413/miniimagenet')
extract_path = Path('/content/miniimagenet')
extract_path.mkdir(parents=True, exist_ok=True)

for split in ['train', 'val', 'test']:
    split_dir = extract_path / split
    tar_path = base_path / f'{split}.tar'
    if not split_dir.exists():
        with tarfile.open(tar_path, 'r') as tar:
            tar.extractall(extract_path)

train_dir = extract_path / 'train'
val_dir = extract_path / 'val'
test_dir = extract_path / 'test'

print("train_dir exists:", train_dir.exists())
print("val_dir exists:", val_dir.exists())
print("test_dir exists:", test_dir.exists())

Mounted at /content/drive


/tmp/ipykernel_15922/1710582717.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_path)


train_dir exists: True
val_dir exists: True
test_dir exists: True


In [5]:
img_size = 96
batch_size = 64
val_frac = 0.10
test_frac = 0.10
num_staff = 2

tfm_train = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tfm_eval = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_train_aug = datasets.ImageFolder(str(train_dir), transform=tfm_train)
ds_train_eval = datasets.ImageFolder(str(train_dir), transform=tfm_eval)

all_idx = list(range(len(ds_train_aug)))
all_targets = ds_train_aug.targets

train_idx, temp_idx = train_test_split(
    all_idx,
    test_size=(val_frac + test_frac),
    stratify=all_targets,
    random_state=seed_orbit
)

temp_targets = [all_targets[i] for i in temp_idx]
valid_idx, test_idx = train_test_split(
    temp_idx,
    test_size=(test_frac / (val_frac + test_frac)),
    stratify=temp_targets,
    random_state=seed_orbit
)

set_train = Subset(ds_train_aug, train_idx)
set_valid = Subset(ds_train_eval, valid_idx)
set_test = Subset(ds_train_eval, test_idx)

ldr_train = DataLoader(set_train, batch_size=batch_size, shuffle=True, num_workers=num_staff, pin_memory=True)
ldr_valid = DataLoader(set_valid, batch_size=batch_size, shuffle=False, num_workers=num_staff, pin_memory=True)
ldr_test = DataLoader(set_test, batch_size=batch_size, shuffle=False, num_workers=num_staff, pin_memory=True)

class_galaxy = ds_train_aug.classes
n_classes = len(class_galaxy)

print(f"#train={len(set_train)} | #val={len(set_valid)} | #test={len(set_test)} | #classes={n_classes}")
print("first 5 classes:", class_galaxy[:5])

#train=30720 | #val=3840 | #test=3840 | #classes=64
first 5 classes: ['n01532829', 'n01558993', 'n01704323', 'n01749939', 'n01770081']


## Model Selection

In [6]:
epoch_budget = 8
lr_res = 3e-4
lr_mob = 4e-4
wd_common = 1e-4
stop_patience = 3
loss_oracle = nn.CrossEntropyLoss()

In [7]:
def train_model(model, optimizer, scheduler, name):
    best_val = 0.0
    best_state = None
    patience_counter = 0
    history = []

    for ep in range(1, epoch_budget + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in ldr_train:
            xb = xb.to(run_device, non_blocking=True)
            yb = yb.to(run_device, non_blocking=True)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_oracle(logits, yb)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * xb.size(0)
            train_correct += (logits.argmax(dim=1) == yb).sum().item()
            train_total += yb.size(0)

        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for xb, yb in ldr_valid:
                xb = xb.to(run_device, non_blocking=True)
                yb = yb.to(run_device, non_blocking=True)
                logits = model(xb)
                loss = loss_oracle(logits, yb)
                val_loss_sum += loss.item() * xb.size(0)
                val_correct += (logits.argmax(dim=1) == yb).sum().item()
                val_total += yb.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        scheduler.step(val_acc)

        history.append({"epoch": ep, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc, "lr_now": optimizer.param_groups[0]["lr"]})
        print(f"[{name}] ep {ep:02d} | tr_loss {train_loss:.4f} tr_acc {train_acc:.4f} | val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= stop_patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return pd.DataFrame(history)

def evaluate_classifier(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    total_loss = 0.0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(run_device, non_blocking=True)
            yb = yb.to(run_device, non_blocking=True)
            logits = model(xb)
            loss = loss_oracle(logits, yb)
            preds = logits.argmax(dim=1)
            total_loss += loss.item() * xb.size(0)
            total += yb.size(0)
            y_true.extend(yb.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return total_loss / total, acc, f1, y_true, y_pred

In [ ]:
resnet_star = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_star.fc = nn.Linear(resnet_star.fc.in_features, n_classes)
resnet_star = resnet_star.to(run_device)
opt_res = optim.AdamW(resnet_star.parameters(), lr=lr_res, weight_decay=wd_common)
sch_res = optim.lr_scheduler.ReduceLROnPlateau(opt_res, mode='max', factor=0.5, patience=1)
res_history_df = train_model(resnet_star, opt_res, sch_res, "ResNet18")
res_loss, res_acc, res_f1, res_true, res_pred = evaluate_classifier(resnet_star, ldr_test)
print("ResNet18 test_loss:", round(res_loss, 4))
print("ResNet18 test_acc:", round(res_acc, 4))
print("ResNet18 macro_f1:", round(res_f1, 4))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 31.2MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
mobile_pulse = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
mobile_pulse.classifier[-1] = nn.Linear(mobile_pulse.classifier[-1].in_features, n_classes)
mobile_pulse = mobile_pulse.to(run_device)
opt_mob = optim.AdamW(mobile_pulse.parameters(), lr=lr_mob, weight_decay=wd_common)
sch_mob = optim.lr_scheduler.ReduceLROnPlateau(opt_mob, mode='max', factor=0.5, patience=1)
mob_history_df = train_model(mobile_pulse, opt_mob, sch_mob, "MobileNetV3-Small")
mob_loss, mob_acc, mob_f1, mob_true, mob_pred = evaluate_classifier(mobile_pulse, ldr_test)
print("MobileNetV3-Small test_loss:", round(mob_loss, 4))
print("MobileNetV3-Small test_acc:", round(mob_acc, 4))
print("MobileNetV3-Small macro_f1:", round(mob_f1, 4))

## Wavelet Compression

In [ ]:
compression_ratios = [2, 5, 10]
wavelet_out_dir = Path('/content/drive/MyDrive/EE413/part2_wavelet_outputs')
wavelet_out_dir.mkdir(parents=True, exist_ok=True)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize_tensor(x):
    x = x.detach().cpu()
    return (x * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

def tensor_to_display_image(x):
    return denormalize_tensor(x).permute(1, 2, 0).numpy()

def wavelet_compress(img_tensor, ratio=5, wavelet="haar", level=2):
    device = img_tensor.device
    dtype = img_tensor.dtype
    img_np = img_tensor.detach().cpu().float().numpy()
    compressed = np.zeros_like(img_np, dtype=np.float32)

    for c in range(img_np.shape[0]):
        channel = img_np[c]
        coeffs = pywt.wavedec2(channel, wavelet=wavelet, level=level)
        coeff_array, coeff_slices = pywt.coeffs_to_array(coeffs)
        flat_abs = np.abs(coeff_array).ravel()
        num_keep = max(1, int(flat_abs.size / ratio))
        threshold = np.partition(flat_abs, -num_keep)[-num_keep]
        compressed_array = coeff_array * (np.abs(coeff_array) >= threshold)
        compressed_coeffs = pywt.array_to_coeffs(compressed_array, coeff_slices, output_format="wavedec2")
        reconstructed = pywt.waverec2(compressed_coeffs, wavelet=wavelet)
        reconstructed = reconstructed[:channel.shape[0], :channel.shape[1]]
        compressed[c] = reconstructed

    return torch.tensor(compressed, dtype=dtype, device=device).clamp(-5, 5)

class WaveletCompressedDataset(Dataset):
    def __init__(self, base_dataset, ratio=5, wavelet="haar", level=2):
        self.base_dataset = base_dataset
        self.ratio = ratio
        self.wavelet = wavelet
        self.level = level

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        return wavelet_compress(img, ratio=self.ratio, wavelet=self.wavelet, level=self.level), label

def per_class_accuracy(y_true, y_pred, class_names):
    rows = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    for idx, name in enumerate(class_names):
        mask = y_true == idx
        samples = int(mask.sum())
        acc = float((y_pred[mask] == y_true[mask]).mean()) if samples > 0 else np.nan
        rows.append({"Class": name, "Class_Index": idx, "Samples": samples, "Accuracy": acc})

    return pd.DataFrame(rows)

In [ ]:
example_indices = sorted(set([0, min(50, len(set_test) - 1), min(150, len(set_test) - 1)]))

for sample_idx in example_indices:
    original_img, label = set_test[sample_idx]
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 4, 1)
    plt.imshow(tensor_to_display_image(original_img))
    plt.title(f"Original
Class: {class_galaxy[label]}")
    plt.axis("off")

    for j, ratio in enumerate(compression_ratios):
        compressed_img = wavelet_compress(original_img, ratio=ratio)
        plt.subplot(1, 4, j + 2)
        plt.imshow(tensor_to_display_image(compressed_img))
        plt.title(f"Wavelet {ratio}:1
Kept {100 / ratio:.0f}%")
        plt.axis("off")

    plt.suptitle(f"Wavelet Compression Example {sample_idx}")
    plt.tight_layout()
    plt.savefig(wavelet_out_dir / f"wavelet_visual_example_{sample_idx}.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
quality_eval_limit = min(500, len(set_test))
quality_rows = []

for ratio in compression_ratios:
    psnr_values = []
    ssim_values = []

    for i in range(quality_eval_limit):
        original_img, _ = set_test[i]
        compressed_img = wavelet_compress(original_img, ratio=ratio)
        original_np = tensor_to_display_image(original_img)
        compressed_np = tensor_to_display_image(compressed_img)
        psnr_values.append(peak_signal_noise_ratio(original_np, compressed_np, data_range=1.0))
        ssim_values.append(structural_similarity(original_np, compressed_np, channel_axis=-1, data_range=1.0))

    quality_rows.append({
        "Compression": f"{ratio}:1",
        "Coefficients Kept": f"{100 / ratio:.0f}%",
        "Average PSNR": round(float(np.mean(psnr_values)), 3),
        "Average SSIM": round(float(np.mean(ssim_values)), 4),
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)
quality_df.to_csv(wavelet_out_dir / "wavelet_quality_psnr_ssim.csv", index=False)

In [ ]:
models_to_test = [("ResNet18", resnet_star), ("MobileNetV3-Small", mobile_pulse)]
results = []
eval_cache = {}

for model_name, model in models_to_test:
    loss, acc, f1, y_true, y_pred = evaluate_classifier(model, ldr_test)
    results.append({
        "Model": model_name,
        "Test Set": "Original",
        "Compression Ratio": "None",
        "Coefficients Kept": "100%",
        "Loss": round(loss, 4),
        "Accuracy": round(acc * 100, 2),
        "Macro F1": round(f1 * 100, 2),
        "Accuracy Drop": 0.00,
    })
    eval_cache[(model_name, "Original")] = {"y_true": y_true, "y_pred": y_pred, "accuracy": acc}

for ratio in compression_ratios:
    compressed_dataset = WaveletCompressedDataset(set_test, ratio=ratio, wavelet="haar", level=2)
    compressed_loader = DataLoader(compressed_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    for model_name, model in models_to_test:
        loss, acc, f1, y_true, y_pred = evaluate_classifier(model, compressed_loader)
        original_acc = eval_cache[(model_name, "Original")]["accuracy"]
        results.append({
            "Model": model_name,
            "Test Set": f"Wavelet {ratio}:1",
            "Compression Ratio": f"{ratio}:1",
            "Coefficients Kept": f"{100 / ratio:.0f}%",
            "Loss": round(loss, 4),
            "Accuracy": round(acc * 100, 2),
            "Macro F1": round(f1 * 100, 2),
            "Accuracy Drop": round((original_acc - acc) * 100, 2),
        })
        eval_cache[(model_name, f"Wavelet {ratio}:1")] = {"y_true": y_true, "y_pred": y_pred, "accuracy": acc}

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(wavelet_out_dir / "wavelet_accuracy_f1_results.csv", index=False)

In [ ]:
plot_order = ["Original", "Wavelet 2:1", "Wavelet 5:1", "Wavelet 10:1"]
plot_df = results_df.copy()
plot_df["Test Set"] = pd.Categorical(plot_df["Test Set"], categories=plot_order, ordered=True)
plot_df = plot_df.sort_values(["Model", "Test Set"])

plt.figure(figsize=(8, 5))
for model_name in plot_df["Model"].unique():
    model_df = plot_df[plot_df["Model"] == model_name]
    plt.plot(model_df["Test Set"].astype(str), model_df["Accuracy"], marker="o", label=model_name)
plt.xlabel("Test Set")
plt.ylabel("Accuracy (%)")
plt.title("Classification Accuracy vs Wavelet Compression")
plt.xticks(rotation=25)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(wavelet_out_dir / "wavelet_accuracy_plot.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(quality_df["Compression"], quality_df["Average PSNR"], marker="o")
plt.xlabel("Compression Ratio")
plt.ylabel("Average PSNR")
plt.title("PSNR vs Wavelet Compression")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(wavelet_out_dir / "wavelet_psnr_plot.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(quality_df["Compression"], quality_df["Average SSIM"], marker="o")
plt.xlabel("Compression Ratio")
plt.ylabel("Average SSIM")
plt.title("SSIM vs Wavelet Compression")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(wavelet_out_dir / "wavelet_ssim_plot.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
target_model = "ResNet18"
original_class_df = per_class_accuracy(eval_cache[(target_model, "Original")]["y_true"], eval_cache[(target_model, "Original")]["y_pred"], class_galaxy)
compressed_class_df = per_class_accuracy(eval_cache[(target_model, "Wavelet 10:1")]["y_true"], eval_cache[(target_model, "Wavelet 10:1")]["y_pred"], class_galaxy)

affected_df = original_class_df.merge(compressed_class_df, on=["Class", "Class_Index", "Samples"], suffixes=(" Original", " 10:1"))
affected_df["Original Accuracy"] = affected_df["Accuracy Original"] * 100
affected_df["10:1 Accuracy"] = affected_df["Accuracy 10:1"] * 100
affected_df["Accuracy Drop"] = affected_df["Original Accuracy"] - affected_df["10:1 Accuracy"]
affected_df = affected_df[["Class", "Class_Index", "Samples", "Original Accuracy", "10:1 Accuracy", "Accuracy Drop"]]
affected_df = affected_df.sort_values("Accuracy Drop", ascending=False).reset_index(drop=True)

affected_slide_df = affected_df.head(10).copy()
for col in ["Original Accuracy", "10:1 Accuracy", "Accuracy Drop"]:
    affected_slide_df[col] = affected_slide_df[col].round(2)

display(affected_slide_df)
affected_slide_df.to_csv(wavelet_out_dir / "wavelet_most_affected_classes.csv", index=False)

In [ ]:
for path in sorted(wavelet_out_dir.iterdir()):
    print(path)